# Part 0 : Load modules and dataset

**!!  This part has been completed for you for guidance  !!**

In [66]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn import clone
from sklearn import set_config
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import RandomUnderSampler


from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
)

set_config(transform_output = "pandas")
pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

In [67]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [68]:
# TODO - change this to the path where your dataset is uploaded
CD_additional_df = pd.read_csv('/content/drive/MyDrive/Data/CD_additional_modified.csv')

In [69]:
# Summary of the columns
CD_additional_df.describe(include = 'all')

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
count,4119.000000,4119,4119,4119,4119,4119,4119,4119,4119,4119,4119.000000,4119.000000,4119.000000,4119.000000,4119,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119
unique,NaN,12,4,8,3,3,3,2,10,5,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,2
top,NaN,admin.,married,university.degree,no,yes,no,cellular,may,thu,NaN,NaN,NaN,NaN,nonexistent,NaN,NaN,NaN,NaN,NaN,no
freq,NaN,1012,2509,1264,3315,2175,3349,2652,1378,860,NaN,NaN,NaN,NaN,3523,NaN,NaN,NaN,NaN,NaN,3668
mean,40.113620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,256.788055,2.537266,960.422190,0.190337,NaN,0.084972,93.579704,-40.499102,3.621356,5166.481695,NaN
std,10.313362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,254.703736,2.568159,191.922786,0.541788,NaN,1.563114,0.579349,4.594578,1.733591,73.667904,NaN
min,18.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000,0.000000,0.000000,NaN,-3.400000,92.201000,-50.800000,0.635000,4963.600000,NaN
25%,32.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,103.000000,1.000000,999.000000,0.000000,NaN,-1.800000,93.075000,-42.700000,1.334000,5099.100000,NaN
50%,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181.000000,2.000000,999.000000,0.000000,NaN,1.100000,93.749000,-41.800000,4.857000,5191.000000,NaN
75%,47.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,317.000000,3.000000,999.000000,0.000000,NaN,1.400000,93.994000,-36.400000,4.961000,5228.100000,NaN


# Part I - Train using imbalanced dataset

**!!  This part has been completed for you for guidance  !!**

Data preparation, train-test split, cv strategy declaration

In [70]:
# Separate out target as y
y = CD_additional_df['y']

# Drop y. Also drop 'duration' due to data leakage issues (we are building a
# model to predict who to call, we should not use a column whose information
# will not be available at the time of making the prediction)
X = CD_additional_df.drop(columns = ['y' , 'duration'])

# create a new categorical column to code pdays = 999 and replace 999 as 0
X['pdays_new'] = X['pdays'] >= 999
X['pdays'] = X['pdays'].replace(999, 0)

# Do a train-test split right away
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

# Fix the Cross-Validation strategy object as 10-fold and the random seed
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

**!!  This part has been completed for you for guidance  !!**

Preprocessing pipelines for tree ensembles (`preprocessor_tree`) and logistic regression (`preprocessor_regr`)
- For tree models we do not need to scale numeric columns, and we keep all categorical one-hot-encoded columns
- For logistic regression with regularization, we standardize numeric columns and for every categorical column we drop the first one-hot-encoded level as reference level

In [71]:
numeric_features = make_column_selector(dtype_include=["number"])
categorical_features = make_column_selector(dtype_include=["object", "category", "bool"])

# Transformer for Decision Trees -- we do not drop columns for categorical and do not need to scale numeric columns
preprocessor_tree = ColumnTransformer(
    transformers= [('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown ='ignore', sparse_output=False), categorical_features)
    ],
    remainder = 'drop',
    verbose_feature_names_out = False
)

# Transformer for Regression
preprocessor_regr = ColumnTransformer(
    transformers= [('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown ='ignore', drop='first', sparse_output=False), categorical_features)
    ],
    remainder = 'drop',
    verbose_feature_names_out = False
)

## Part I.a - Defining and Cross-validating Gradient Boosted Trees

**!!  This part has been completed for you for guidance  !!**


In [72]:
# Training and Cross-validating Gradient Boosted Trees

# list of number of trees in the Gradient Boosting ensemble -- we will pick the
# best from this list using cross-validation
n_estimator_vec = [10, 30, 40, 50, 100, 150, 200]

# defining the gradient boosting classifier
gb_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE, # setting the random seed
    n_estimators=200, # this value will be reset inside the loop below
    learning_rate=0.025, # smaller learning rate means each additional member of ensemble has less impact
    max_depth=3, # maximum depth of each member tree in the ensemble
)

# pipeling of data preprocessing + gradient boosting classifier
pipe_gb_imbalanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", gb_model),
])

# here we loop over values in n_estimator_vec, use 10-fold cross validation to
# find the cross validation estimate of test score (here we are using auc score
# as the metric to choose the best value of n_estimators)
best_score = 0
best_n_estimators = 0
for n_estimators in n_estimator_vec:
    pipe_gb_imbalanced.set_params(model__n_estimators=n_estimators)
    mean_auc = cross_val_score(pipe_gb_imbalanced, X_train, y_train, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_n_estimators = n_estimators
    print(f"n_estimators: {n_estimators}, mean auc: {mean_auc}")


print(f"Best n_estimators: {best_n_estimators}, best mean auc: {best_score}")

n_estimators: 10, mean auc: 0.7466234963916095
n_estimators: 30, mean auc: 0.7556264791851348
n_estimators: 40, mean auc: 0.7586254083004503
n_estimators: 50, mean auc: 0.7553076070749851
n_estimators: 100, mean auc: 0.7655171906318152
n_estimators: 150, mean auc: 0.7666855752819737
n_estimators: 200, mean auc: 0.7659110202265007
Best n_estimators: 150, best mean auc: 0.7666855752819737


**!!  This part has been completed for you for guidance  !!**

Training another Gradient Boosted tree classifier for the `best_n_estimators` found above on the full training data

In [73]:
# Train a new gradient boosted ensemble
# First set the hyperparameter for n_estimators to the value we found above
pipe_gb_imbalanced.set_params(model__n_estimators=best_n_estimators)

# Fit the ensemble on the full trainining data
pipe_gb_imbalanced.fit(X_train, y_train)

# Make predictions on the test split
y_pred = pipe_gb_imbalanced.predict(X_test)
y_pred_proba = pipe_gb_imbalanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_tree_imbalanced = roc_auc_score(y_test, y_pred_proba)
f1_score_tree_imbalanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_tree_imbalanced}")
print(f"F1 Score: {f1_score_tree_imbalanced}")

AUC Score: 0.799518955831399
F1 Score: 0.327683615819209


## Part I.b - Defining and Cross-validating Random Forests

**TODO** : The first half has been completed for you, the second part you have to complete

In [74]:
# Training and Cross-validating Random Forests
n_estimator_vec = [100, 200, 300, 400, 500]

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    oob_score=True,
    max_features="sqrt",   # default in sklearn for classification
)

pipe_rf_imbalanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", rf_model),
])

# ==========================================================================
# TODO -- follow the code in gradient boosted classifier to find the best
# n_estimator using cross validation
# ==========================================================================

best_score = 0
best_n_estimators = 0
for n_estimators in n_estimator_vec:
    pipe_rf_imbalanced.set_params(model__n_estimators=n_estimators)
    mean_auc = cross_val_score(pipe_rf_imbalanced, X_train, y_train, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_n_estimators = n_estimators
    print(f"n_estimators: {n_estimators}, mean auc: {mean_auc}")


print(f"Best n_estimators: {best_n_estimators}, best mean auc: {best_score}")

n_estimators: 100, mean auc: 0.7444010199605109
n_estimators: 200, mean auc: 0.7443652293187425
n_estimators: 300, mean auc: 0.7467551245628942
n_estimators: 400, mean auc: 0.7467269328631974
n_estimators: 500, mean auc: 0.7450071510802372
Best n_estimators: 300, best mean auc: 0.7467551245628942


**TODO**: Training another random forest for the `best_n_estimators` found above

In [75]:
# ==========================================================================
# TODO -- 1. follow the code in gradient boosted classifier to train another random
# forest model on the full train split with the best hyperparameter choice you
# found 2. report the AUC and F1 score on the test split
# ==========================================================================

# Train a new random forest ensemble
# First set the hyperparameter for n_estimators to the value we found above
pipe_rf_imbalanced.set_params(model__n_estimators=best_n_estimators)

# Fit the ensemble on the full training data
pipe_rf_imbalanced.fit(X_train, y_train)

# Make predictions on the test split
y_pred = pipe_rf_imbalanced.predict(X_test)
y_pred_proba = pipe_rf_imbalanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_rf_imbalanced = roc_auc_score(y_test, y_pred_proba)
f1_score_rf_imbalanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_rf_imbalanced}")
print(f"F1 Score: {f1_score_rf_imbalanced}")

AUC Score: 0.7658929592626234
F1 Score: 0.32978723404255317


## Part I.c - Defininig and Cross-validating Logistic regression with Ridge (l2) penalty

**TODO** : The first half has been completed for you, the second part you have to complete

In [76]:
# For sklearn logistic regression, C is the inverse of the penalty knob -- low C means high penalization
# List of C values to try and pick the best from using cross-validation
Cs_vec = np.logspace(np.log10(0.001), np.log10(10), 10)

logit_model = LogisticRegression(penalty='l2')

pipe_logit_imbalanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_regr)),
    ("model", logit_model)])

# ==========================================================================
# TODO -- follow the code in gradient boosted classifier to find the best
# C value using cross validation
# Hint: you can set the C value using
#        pipe_logit_imbalanced.set_params(model__C = new_value)
# ==========================================================================

best_score = 0
best_C = 0
for C in Cs_vec:
    pipe_logit_imbalanced.set_params(model__C=C)
    mean_auc = cross_val_score(pipe_logit_imbalanced, X_train, y_train, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_C = C
    print(f"C: {C}, mean auc: {mean_auc}")


print(f"Best C: {best_C}, best mean auc: {best_score}")

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 0.001, mean auc: 0.7526660266568345


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 0.0027825594022071257, mean auc: 0.7526124640362196


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


...

C: 1.2915496650148828, mean auc: 0.7533845606780313


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 3.593813663804626, mean auc: 0.7507928487481564


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 10.0, mean auc: 0.7486723861416624
Best C: 0.05994842503189409, best mean auc: 0.7628386037718087


**TODO** - Training another Logistic Regression with optimal Ridge penalty parameter `best_Cs` found above on the full training split

In [77]:
# ==========================================================================
# TODO -- 1. follow the code in gradient boosted classifier to train another
# logistic regression model with ridge penalty on the full train split with the
# best hyperparameter choice you found 2. report the AUC and F1 score on the
# test split
# ==========================================================================\

# Train a new logistic regression model (ridge / L2)
# First set the hyperparameter C to the best value found above
pipe_logit_imbalanced.set_params(model__C=best_C)

# Fit the model on the full training data
pipe_logit_imbalanced.fit(X_train, y_train)

# Make predictions on the test split
y_pred = pipe_logit_imbalanced.predict(X_test)
y_pred_proba = pipe_logit_imbalanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_logit_imbalanced = roc_auc_score(y_test, y_pred_proba)
f1_score_logit_imbalanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_logit_imbalanced}")
print(f"F1 Score: {f1_score_logit_imbalanced}")

AUC Score: 0.7876946883304741
F1 Score: 0.23170731707317074


# Part II - Train using balanced dataset

**!!  This part has been completed for you for guidance  !!**

Creating a balanced subsample out of training split

In [78]:
# Here we are creating a further subsampled training data from X_train and Y_train
# The new subsampled data is class balanced
rus=RandomUnderSampler(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = rus.fit_resample(X_train, y_train)
# confirm the class balance below
y_train_balanced.value_counts()

,count
y,
no,316
yes,316


## Part II.a - Defining and Cross-validating Gradient Boosted Trees


**TODO**

In [79]:
#===============================================================================
# TODO - Use the code in Part I.a to do hyperparameter optimization for Gradient
# Boosted trees, using ONLY the X_train_balanced, y_train_balanced data
#===============================================================================

# Training and Cross-validating Gradient Boosted Trees

# list of number of trees in the Gradient Boosting ensemble -- we will pick the
# best from this list using cross-validation
n_estimator_vec = [10, 30, 40, 50, 100, 150, 200]

# defining the gradient boosting classifier
gb_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE, # setting the random seed
    n_estimators=200, # this value will be reset inside the loop below
    learning_rate=0.025, # smaller learning rate means each additional member of ensemble has less impact
    max_depth=3, # maximum depth of each member tree in the ensemble
)

# pipeling of data preprocessing + gradient boosting classifier
pipe_gb_balanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", gb_model),
])

# here we loop over values in n_estimator_vec, use 10-fold cross validation to
# find the cross validation estimate of test score (here we are using auc score
# as the metric to choose the best value of n_estimators)
best_score = 0
best_n_estimators = 0
for n_estimators in n_estimator_vec:
    pipe_gb_balanced.set_params(model__n_estimators=n_estimators)
    mean_auc = cross_val_score(pipe_gb_balanced, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_n_estimators = n_estimators
    print(f"n_estimators: {n_estimators}, mean auc: {mean_auc}")


print(f"Best n_estimators (balanced data): {best_n_estimators}, best mean auc (balanced data): {best_score}")

n_estimators: 10, mean auc: 0.7555931829637096
n_estimators: 30, mean auc: 0.7454999369959677
n_estimators: 40, mean auc: 0.7439027847782258
n_estimators: 50, mean auc: 0.7423009072580646
n_estimators: 100, mean auc: 0.7398232736895162
n_estimators: 150, mean auc: 0.7357563634072581
n_estimators: 200, mean auc: 0.7355200982862903
Best n_estimators (balanced data): 10, best mean auc (balanced data): 0.7555931829637096


**TODO** - Training another Gradient Boosted tree classifier for `best_n_estimators` found above on `X_train_balanced`

In [80]:
#===============================================================================
# TODO - Use the code in Part I.a to do train another Gradient Boosted ensemble
# on the ENTIRE X_train_balanced, y_train_balanced data for the optimal choice
# of hyperparameter you found above. Report AUC and F1 score on the original test split
# ===============================================================================

# Train a new gradient boosted ensemble
# First set the hyperparameter for n_estimators to the value we found above
pipe_gb_balanced.set_params(model__n_estimators=best_n_estimators)

# Fit the ensemble on the balanced training data
pipe_gb_balanced.fit(X_train_balanced, y_train_balanced)

# Make predictions on the test split
y_pred = pipe_gb_balanced.predict(X_test)
y_pred_proba = pipe_gb_balanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_tree_balanced = roc_auc_score(y_test, y_pred_proba)
f1_score_tree_balanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_tree_balanced}")
print(f"F1 Score: {f1_score_tree_balanced}")

AUC Score: 0.7526894742153597
F1 Score: 0.3956639566395664


## Part II.b - Defining and Cross-validating Random Forests

**TODO**

In [81]:
#===============================================================================
# TODO - Use the code in Part I.b to do hyperparameter optimization for Random
# Forest, using ONLY the X_train_balanced, y_train_balanced data
#===============================================================================

# Training and Cross-validating Random Forests
n_estimator_vec = [100, 200, 300, 400, 500]

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    oob_score=True,
    max_features="sqrt",   # default in sklearn for classification
)

pipe_rf_balanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_tree)),
    ("model", rf_model),
])

best_score = 0
best_n_estimators = 0
for n_estimators in n_estimator_vec:
    pipe_rf_balanced.set_params(model__n_estimators=n_estimators)
    mean_auc = cross_val_score(pipe_rf_balanced, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_n_estimators = n_estimators
    print(f"n_estimators: {n_estimators}, mean auc: {mean_auc}")


print(f"Best n_estimators (balanced data): {best_n_estimators}, best mean auc (balanced data): {best_score}")

n_estimators: 100, mean auc: 0.7359926285282258
n_estimators: 200, mean auc: 0.7398311491935484
n_estimators: 300, mean auc: 0.7405871975806451
n_estimators: 400, mean auc: 0.7406186995967742
n_estimators: 500, mean auc: 0.7400941910282258
Best n_estimators (balanced data): 400, best mean auc (balanced data): 0.7406186995967742


**TODO** - Training another random forest for the `best_n_estimators` found above

In [82]:
#===============================================================================
# TODO - Use the code in Part I.b to do train another Random Forest ensemble
# on the ENTIRE X_train_balanced, y_train_balanced data for the optimal choice
# of hyperparameter you found above. Report AUC and F1 score on the original test split
#===============================================================================

# Train a new random forest ensemble
# First set the hyperparameter for n_estimators to the value we found above
pipe_rf_balanced.set_params(model__n_estimators=best_n_estimators)

# Fit the ensemble on the balanced training data
pipe_rf_balanced.fit(X_train_balanced, y_train_balanced)

# Make predictions on the test split
y_pred = pipe_rf_balanced.predict(X_test)
y_pred_proba = pipe_rf_balanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_rf_balanced = roc_auc_score(y_test, y_pred_proba)
f1_score_rf_balanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_rf_balanced}")
print(f"F1 Score: {f1_score_rf_balanced}")

AUC Score: 0.7568809499781343
F1 Score: 0.36936936936936937


## Part II.c - Defining and Cross-validating Logistic Regression with Ridge (l2) penalty

**TODO**

In [83]:
#===============================================================================
# TODO - Use the code in Part I.c to do hyperparameter optimization for Logistic
# Regression, using ONLY the X_train_balanced, y_train_balanced data
#===============================================================================

# For sklearn logistic regression, C is the inverse of the penalty knob -- low C means high penalization
# List of C values to try and pick the best from using cross-validation
Cs_vec = np.logspace(np.log10(0.001), np.log10(10), 10)

logit_model = LogisticRegression(penalty='l2')

pipe_logit_balanced = Pipeline(steps=[
    ("preprocess", clone(preprocessor_regr)),
    ("model", logit_model)])

best_score = 0
best_C = 0
for C in Cs_vec:
    pipe_logit_balanced.set_params(model__C=C)
    mean_auc = cross_val_score(pipe_logit_balanced, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring="roc_auc").mean()
    if mean_auc > best_score:
        best_score = mean_auc
        best_C = C
    print(f"C: {C}, mean auc: {mean_auc}")


print(f"Best C: {best_C}, best mean auc: {best_score}")

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 0.001, mean auc: 0.7456180695564517


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 0.0027825594022071257, mean auc: 0.7459047379032258


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


...

C: 1.2915496650148828, mean auc: 0.7405115927419355


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 3.593813663804626, mean auc: 0.7388671874999999


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C: 10.0, mean auc: 0.7397397933467741
Best C: 0.05994842503189409, best mean auc: 0.7482327368951612


**TODO** - Training another Logistic Regression classifier for `best_Cs` found above on `X_train_balanced`

In [84]:
#===============================================================================
# TODO - Use the code in Part I.c to do train another Logistic Regression model
# on the ENTIRE X_train_balanced, y_train_balanced data for the optimal choice
# of hyperparameter you found above. Report AUC and F1 score on the original test split
# ===============================================================================

# Train a new logistic regression model (ridge / L2)
# First set the hyperparameter C to the best value found above
pipe_logit_balanced.set_params(model__C=best_C)

# Fit the model on the balanced training data
pipe_logit_balanced.fit(X_train_balanced, y_train_balanced)

# Make predictions on the test split
y_pred = pipe_logit_balanced.predict(X_test)
y_pred_proba = pipe_logit_balanced.predict_proba(X_test)[:, 1]

# Report AUC and F1 score on the test split
auc_score_logit_balanced = roc_auc_score(y_test, y_pred_proba)
f1_score_logit_balanced = f1_score(y_test, y_pred, pos_label='yes')
print(f"AUC Score: {auc_score_logit_balanced}")
print(f"F1 Score: {f1_score_logit_balanced}")

AUC Score: 0.7755172065798769
F1 Score: 0.37986270022883295


# Part III - Targeting customers

The company can only investigate a limited number of cases. They would like to use the classification models learned above to rank customers based on the predicted probability of purchase, and target the top few.

Assume the company has an operating budget to only target the top 10% of cases based on the probability of purchase (probability of label y = True).
Using the untouched test set from the original imbalanced data, for each model report how many actual positives appear in the selected group (the top 10%).



**TODO** - a template code is provided. Use that as a guide to complete this code block

In [85]:
model = pipe_gb_imbalanced

# find the predicted probability of y=True for all rows in test split
probs = model.predict_proba(X_test)[:, 1]
# k is our targeting budget == 10% of the customers in test split
k = int(0.10 * len(probs))
# sort customers by the predicted purchase probability
top_idx = np.argsort(probs)[::-1][:k]
# find the true lable of the customer identified above
y_top = y_test.iloc[top_idx]
# Find what fraction of the customers we targeted (in top_idx) have true label of 'yes'
precision_at_k = y_top.eq('yes').mean()

#===============================================================================
# TODO - use the template above to find the precision at 10% for all the 6
# models you created and present the results as a dataframe
#===============================================================================


# Part IV - Interpretation and recommendations

Interpret your results from Parts I-III and answer the following questions. Building models is useful, but understanding why they behave the way they do is what gives you an advantage.

1. Did the model with the best cross-validated ROC-AUC also achieve the best precision at 10% on the test set?
If not, explain why a model that ranks observations well overall may still not be the best model for a targeting task.
2. For a given model class, how did training on the balanced dataset versus the original imbalanced dataset affect performance in the top-10% targeting exercise?
Based on your results, did undersampling appear to help by making positives easier to identify, or hurt by discarding useful information from the majority class?
3. How did the tree-based models compare with logistic regression in the targeting exercise?
What kinds of relationships might flexible tree ensembles capture that logistic regression may miss?
4. Suppose the bank can contact only 10% of customers. Which model would you recommend, and why?
Your answer should refer to both the cross-validation results and the final targeting results on the test set.

**Bonus**

5. Try a few different values of `RANDOM_STATE`. How robust are your conclusions?
In particular, does the ordering of the models in the top-10% targeting exercise remain similar?
6. What are some other ways to train models when the data are imbalanced?